# Task C — SOC Automation System

This notebook extends the benchmarking (Task A) and analyst tooling (Task B) notebooks by orchestrating an automated SOC (Security Operations Center) workflow. It simulates a continuous phishing triage pipeline, enriches alerts with contextual intelligence, stores incidents, and visualizes emerging threats for a master's thesis demonstration.

## 1️⃣ Setup & Imports

The first cell installs any dependencies that are not preloaded in the Kaggle runtime. Rerun it if you restart the kernel.

In [ ]:
%%capture
!pip install --quiet pandas numpy scikit-learn joblib python-whois requests validators tqdm matplotlib seaborn

In [ ]:
import os
import json
import math
import random
import sqlite3
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
import validators

try:
    import whois
except Exception:
    whois = None  # Handle environments without native dependencies

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

REPORTS_DIR = Path("/kaggle/working/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR_CANDIDATES = [
    Path("/kaggle/working/artifacts/best_model"),
    Path("/kaggle/input/phishing-detection-artifacts"),
]
DB_PATH = Path("/kaggle/working/soc_incidents.db")
SIM_DATA_PATH = DATA_DIR / "sim_emails.csv"


## 2️⃣ Load Best Model (from Task A artifacts)

The helper below searches common Kaggle locations (`/kaggle/working/artifacts/best_model` or an attached dataset) for the serialized model, vectorizer, and preprocessing configuration generated in Task A.

In [ ]:
def locate_artifacts() -> Optional[Path]:
    for cand in ARTIFACT_DIR_CANDIDATES:
        if cand.exists():
            if (cand / "model.joblib").exists() and (cand / "vectorizer.joblib").exists():
                return cand
            if (cand / "config.json").exists():
                return cand
    raise FileNotFoundError(
        "Could not locate Task A artifacts. Attach the exported bundle as a Kaggle dataset "
        "or run Task A in the same session to populate /kaggle/working/artifacts/best_model/."
    )

ARTIFACT_DIR = locate_artifacts()
print(f"Using artifacts from: {ARTIFACT_DIR}")


In [ ]:
def load_preproc_config(path: Path) -> Dict:
    cfg_path = path / "preproc_config.json"
    if cfg_path.exists():
        with open(cfg_path, "r", encoding="utf-8") as fh:
            return json.load(fh)
    return {
        "lower": True,
        "url_token": "<URL>",
        "num_token": "<NUM>",
        "strip_html": True,
    }

preproc_config = load_preproc_config(ARTIFACT_DIR)
model_path = ARTIFACT_DIR / "model.joblib"
vectorizer_path = ARTIFACT_DIR / "vectorizer.joblib"

if not model_path.exists() or not vectorizer_path.exists():
    raise FileNotFoundError(
        "This notebook currently supports classical models exported from Task A. "
        "Ensure the best_model bundle contains model.joblib and vectorizer.joblib."
    )

model = joblib.load(model_path)
vectorizer: TfidfVectorizer = joblib.load(vectorizer_path)
print("Loaded model:", type(model).__name__)


In [ ]:
import re
from bs4 import BeautifulSoup

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+", re.IGNORECASE)
DOMAIN_PATTERN = re.compile(r"([a-z0-9.-]+\.[a-z]{2,})", re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
NUM_PATTERN = re.compile(r"\d+")


def strip_html(text: str) -> str:
    if not text:
        return ""
    soup = BeautifulSoup(text, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    return soup.get_text(separator=" ", strip=True)


def normalize_text(text: str, lower: bool = True, url_token: str = "<URL>", num_token: str = "<NUM>") -> str:
    if not text:
        return ""
    processed = text
    processed = re.sub(URL_PATTERN, url_token, processed)
    processed = re.sub(NUM_PATTERN, num_token, processed)
    processed = re.sub(r"\s+", " ", processed).strip()
    if lower:
        processed = processed.lower()
    return processed


def preprocess(text: str, is_html: bool = False) -> str:
    clean = strip_html(text) if is_html and preproc_config.get("strip_html", True) else text
    return normalize_text(
        clean,
        lower=preproc_config.get("lower", True),
        url_token=preproc_config.get("url_token", "<URL>"),
        num_token=preproc_config.get("num_token", "<NUM>"),
    )


def classify_message(subject: Optional[str], body: str, is_html: bool = False) -> Tuple[int, float]:
    subject = subject or ""
    merged = f"{subject} {body}".strip()
    clean = preprocess(merged, is_html=is_html)
    vec = vectorizer.transform([clean])
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(vec)[0, 1]
    else:
        score = model.decision_function(vec)
        prob = 1 / (1 + np.exp(-score))[0]
    label = int(prob >= 0.5)
    return label, float(prob)


## 3️⃣ Simulate Email Stream

Create a synthetic inbox with realistic phishing and benign language, ensuring repeatable results. The dataset is saved to `/kaggle/working/data/sim_emails.csv` so that it can be reused or inspected outside this notebook.

In [ ]:
def generate_synthetic_email(idx: int, phishing: bool) -> Dict[str, object]:
    base_time = datetime.utcnow() - timedelta(hours=12)
    timestamp = base_time + timedelta(minutes=idx * 5)

    phishing_templates = [
        (
            "IT Support",
            "URGENT: Password Reset Required",
            "Dear user, your mailbox has exceeded the storage limit. Please click http://secure-login-update.com to reactivate your account.",
        ),
        (
            "Corporate Payroll",
            "Payroll Verification Needed",
            "We detected a discrepancy in your salary account. Verify your information at https://corp-payroll-secure.net immediately.",
        ),
        (
            "Office365 Security",
            "Security Alert",
            "Your account has been accessed from an unknown device. Validate here: http://login-office365-security.com.",
        ),
        (
            "Helpdesk",
            "Important: Two-factor reset",
            "We are upgrading our security. Disable your MFA temporarily using https://helpdesk-mfa-reset.app.",
        ),
    ]

    legitimate_templates = [
        (
            "HR Team",
            "Welcome to the company newsletter",
            "Hello team, here are the highlights for this month including upcoming events and training sessions.",
        ),
        (
            "Project Manager",
            "Sprint demo notes",
            "Thanks everyone for attending the sprint review. Attached are the summary notes and action items.",
        ),
        (
            "Finance",
            "Invoice confirmation",
            "Please find attached the invoice for last quarter's cloud expenses. Let us know if you have any questions.",
        ),
        (
            "Security Team",
            "Weekly phishing stats",
            "Sharing the weekly phishing statistics dashboard for situational awareness. No action needed.",
        ),
    ]

    sender_domains = [
        "example.com",
        "corp.example.com",
        "vendor.net",
        "security-alerts.co",
        "secure-update.io",
        "mailservice.org",
    ]

    if phishing:
        sender = f"alerts@{random.choice(sender_domains)}"
        template = phishing_templates[idx % len(phishing_templates)]
    else:
        sender = f"{random.choice(['announcements', 'info', 'updates', 'billing'])}@{random.choice(sender_domains)}"
        template = legitimate_templates[idx % len(legitimate_templates)]

    subject = template[1]
    body = template[2]
    is_html = random.random() < (0.35 if phishing else 0.2)
    if is_html:
        body = f"<html><body><p>{body}</p><p>Thank you.</p></body></html>"

    return {
        "timestamp": timestamp.isoformat(),
        "sender": sender,
        "subject": subject,
        "body": body,
        "body_is_html": is_html,
        "simulated_label": int(phishing),
    }


def build_simulated_inbox(num_messages: int = 200) -> pd.DataFrame:
    half = num_messages // 2
    entries = []
    for i in range(num_messages):
        phishing = i < half
        entries.append(generate_synthetic_email(i, phishing))
    df = pd.DataFrame(entries)
    df.to_csv(SIM_DATA_PATH, index=False)
    return df


if SIM_DATA_PATH.exists():
    simulated_emails = pd.read_csv(SIM_DATA_PATH)
else:
    simulated_emails = build_simulated_inbox(200)

print(simulated_emails.head())
print(f"Simulated inbox saved to: {SIM_DATA_PATH}")


## 4️⃣ Automated Classification & Alert Generation

Process the synthetic inbox in arrival order, classify each message, and collect phishing detections into an alert queue for enrichment.

In [ ]:
processed_alerts: List[Dict[str, object]] = []
stream_records: List[Dict[str, object]] = []

for _, row in tqdm(simulated_emails.iterrows(), total=len(simulated_emails), desc="Processing inbox"):
    label, score = classify_message(row.get("subject"), row.get("body"), bool(row.get("body_is_html")))
    record = row.to_dict()
    record.update({
        "predicted_label": label,
        "phishing_prob": score,
    })
    stream_records.append(record)
    if label == 1:
        processed_alerts.append(record)

classified_df = pd.DataFrame(stream_records)
alerts_df = pd.DataFrame(processed_alerts)
print(f"Total messages processed: {len(classified_df)}")
print(f"Alerts generated: {len(alerts_df)}")


## 5️⃣ Threat Enrichment Module

Threat enrichment mimics SOC triage by extracting URLs, querying WHOIS metadata (with offline fallbacks), and flagging suspicious patterns such as IP-based URLs, URL shorteners, or uncommon top-level domains.

In [ ]:
SUSPICIOUS_TLDS = {"zip", "top", "kim", "biz", "xyz", "support", "fit", "rest"}
URL_SHORTENERS = {"bit.ly", "tinyurl.com", "goo.gl", "ow.ly", "t.co"}


def extract_urls(text: str) -> List[str]:
    if not text:
        return []
    urls = re.findall(URL_PATTERN, text)
    return list({url.strip('.,;"' ) for url in urls})


def extract_domains(urls: List[str]) -> List[str]:
    domains = []
    for url in urls:
        stripped = url
        if stripped.startswith("http"):
            stripped = stripped.split("//", 1)[-1]
        stripped = stripped.split("/", 1)[0]
        stripped = stripped.split(":", 1)[0]
        if validators.domain(stripped):
            domains.append(stripped.lower())
    return list({d for d in domains if d})


def estimate_domain_age(domain: str) -> Optional[int]:
    if not domain:
        return None
    if whois is None:
        if domain.endswith("example.com"):
            return 3650
        return random.randint(10, 400)
    try:
        w = whois.whois(domain)
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        if not creation_date:
            return None
        if isinstance(creation_date, str):
            creation_date = datetime.fromisoformat(creation_date)
        age = (datetime.utcnow() - creation_date).days
        return max(age, 0)
    except Exception:
        return random.randint(5, 300)


def mock_reputation_score(domain: Optional[str]) -> float:
    if not domain:
        return 75.0
    base = random.uniform(30, 90)
    if any(domain.endswith(tld) for tld in SUSPICIOUS_TLDS):
        base -= 25
    if domain in URL_SHORTENERS:
        base -= 30
    if domain.endswith("example.com"):
        base += 10
    return float(max(1.0, min(100.0, base)))


def count_suspicious_patterns(urls: List[str]) -> int:
    suspicious = 0
    for url in urls:
        domain = url
        if domain.startswith("http"):
            domain = domain.split("//", 1)[-1]
        domain = domain.split("/", 1)[0]
        if validators.ipv4(domain):
            suspicious += 1
        if any(short in domain for short in URL_SHORTENERS):
            suspicious += 1
        if any(domain.endswith(tld) for tld in SUSPICIOUS_TLDS):
            suspicious += 1
    return suspicious


def enrich_alert(record: Dict[str, object]) -> Dict[str, object]:
    body_text = record.get("body", "")
    urls = extract_urls(body_text)
    domains = extract_domains(urls)
    primary_domain = domains[0] if domains else None
    age_days = estimate_domain_age(primary_domain) if primary_domain else None
    reputation = mock_reputation_score(primary_domain)
    suspicious_count = count_suspicious_patterns(urls)

    enriched = record.copy()
    enriched.update({
        "urls": urls,
        "domains": domains,
        "primary_domain": primary_domain,
        "domain_age_days": age_days if age_days is not None else -1,
        "reputation_score": reputation,
        "num_urls": len(urls),
        "num_suspicious_patterns": suspicious_count,
    })
    return enriched


if len(alerts_df) == 0:
    print("No alerts generated. Consider adjusting the classifier threshold or simulation mix.")
else:
    enriched_alerts = [enrich_alert(rec) for rec in alerts_df.to_dict(orient="records")]
    enriched_df = pd.DataFrame(enriched_alerts)
    display(enriched_df.head())


## 6️⃣ Incident Scoring & Prioritization

Combine model confidence, domain reputation, and domain age to compute a final priority score and map to analyst-friendly severity levels with suggested playbooks.

In [ ]:
SEVERITY_THRESHOLDS = {
    "Critical": 0.80,
    "High": 0.65,
    "Medium": 0.45,
}

PLAYBOOKS = {
    "Critical": "Escalate to Tier-2 analyst and isolate affected accounts",
    "High": "Quarantine message and notify targeted user",
    "Medium": "Queue for analyst review within 24 hours",
    "Low": "Log for monitoring; no immediate action",
}


def compute_priority(row: pd.Series) -> Tuple[float, str, str]:
    prob = row.get("phishing_prob", 0.0)
    reputation = row.get("reputation_score", 50.0)
    age_days = row.get("domain_age_days", -1)
    age_component = 0.0
    if age_days is not None and age_days >= 0:
        age_component = 1 / (age_days + 1)
    priority = (prob * 0.6) + ((1 - reputation / 100.0) * 0.3) + (age_component * 0.1)

    severity = "Low"
    for sev, threshold in SEVERITY_THRESHOLDS.items():
        if priority >= threshold:
            severity = sev
            break
    action = PLAYBOOKS.get(severity, PLAYBOOKS["Low"])
    return priority, severity, action


if len(alerts_df) > 0:
    priority_data = []
    for _, row in enriched_df.iterrows():
        priority, severity, action = compute_priority(row)
        priority_data.append({
            **row.to_dict(),
            "priority_score": priority,
            "severity": severity,
            "recommended_action": action,
        })
    prioritized_df = pd.DataFrame(priority_data)
    display(prioritized_df[["timestamp", "sender", "subject", "phishing_prob", "reputation_score", "priority_score", "severity", "recommended_action"]].head())
else:
    prioritized_df = pd.DataFrame()


## 7️⃣ Database Storage

Persist alerts to a SQLite database (`soc_incidents.db`) so historical incidents can be analyzed or exported into SIEM tooling.

In [ ]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS alerts (
  id TEXT PRIMARY KEY,
  timestamp TEXT,
  sender TEXT,
  subject TEXT,
  phishing_prob REAL,
  severity TEXT,
  domain TEXT,
  reputation_score REAL,
  domain_age_days INTEGER,
  num_urls INTEGER,
  action TEXT
)
"""


def upsert_alerts(df: pd.DataFrame, db_path: Path) -> None:
    if df.empty:
        print("No alerts to persist.")
        return
    conn = sqlite3.connect(db_path)
    conn.execute(CREATE_TABLE_SQL)
    insert_sql = """
    INSERT OR REPLACE INTO alerts (
        id, timestamp, sender, subject, phishing_prob, severity,
        domain, reputation_score, domain_age_days, num_urls, action
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    rows = []
    for _, row in df.iterrows():
        alert_id = f"{row.get('timestamp')}::{row.get('sender')}::{row.get('subject')}"
        rows.append((
            alert_id,
            row.get("timestamp"),
            row.get("sender"),
            row.get("subject"),
            float(row.get("phishing_prob", 0.0)),
            row.get("severity", "Low"),
            row.get("primary_domain"),
            float(row.get("reputation_score", 0.0)),
            int(row.get("domain_age_days", -1)),
            int(row.get("num_urls", 0)),
            row.get("recommended_action", "Log"),
        ))
    conn.executemany(insert_sql, rows)
    conn.commit()
    conn.close()
    print(f"Persisted {len(rows)} alerts to {db_path}")


if not prioritized_df.empty:
    upsert_alerts(prioritized_df, DB_PATH)


## 8️⃣ Dashboard & Visualization

Generate situational awareness dashboards showing phishing trends, severity distribution, and enrichment correlations. The primary visualization is saved to `/kaggle/working/reports/soc_dashboard.png`.

In [ ]:
if prioritized_df.empty:
    print("No alerts available for visualization.")
else:
    viz_df = prioritized_df.copy()
    viz_df["timestamp"] = pd.to_datetime(viz_df["timestamp"])
    viz_df.sort_values("timestamp", inplace=True)

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    daily_counts = viz_df.set_index("timestamp").resample("1H").size()
    daily_counts.plot(ax=axes[0, 0], marker="o", color="#d62728")
    axes[0, 0].set_title("Phishing Alerts Over Time")
    axes[0, 0].set_xlabel("Timestamp")
    axes[0, 0].set_ylabel("Alert Count")

    severity_counts = viz_df["severity"].value_counts().reindex(["Critical", "High", "Medium", "Low"], fill_value=0)
    severity_counts.plot(kind="bar", ax=axes[0, 1], color="#1f77b4")
    axes[0, 1].set_title("Severity Distribution")
    axes[0, 1].set_xlabel("Severity")
    axes[0, 1].set_ylabel("Number of Alerts")

    top_domains = viz_df["primary_domain"].fillna("unknown").value_counts().head(10)
    top_domains.plot(kind="barh", ax=axes[1, 0], color="#2ca02c")
    axes[1, 0].invert_yaxis()
    axes[1, 0].set_title("Top Flagged Domains")
    axes[1, 0].set_xlabel("Alert Count")

    scatter = axes[1, 1].scatter(
        viz_df["phishing_prob"],
        viz_df["reputation_score"],
        c=viz_df["priority_score"],
        cmap="viridis",
        alpha=0.7,
    )
    axes[1, 1].set_title("Model Confidence vs Domain Reputation")
    axes[1, 1].set_xlabel("Phishing Probability")
    axes[1, 1].set_ylabel("Reputation Score")
    cbar = fig.colorbar(scatter, ax=axes[1, 1])
    cbar.set_label("Priority Score")

    plt.tight_layout()
    dashboard_path = REPORTS_DIR / "soc_dashboard.png"
    plt.savefig(dashboard_path, dpi=200)
    plt.show()
    print(f"Dashboard saved to: {dashboard_path}")


## 9️⃣ Discussion of SOC Integration

The automated pipeline built above demonstrates how a phishing detection model can underpin a full SOC automation workflow:

* **Reduced analyst workload:** Automated classification and enrichment remove repetitive triage steps so analysts can focus on high-severity incidents.
* **Context-rich alerts:** Domain age, reputation heuristics, and IOC extraction provide immediate context that speeds decision-making.
* **Operational integration:** The SQLite database acts as a lightweight incident repository that could be exported to SIEM platforms (Splunk, ELK) via REST APIs, syslog forwarding, or scheduled ingestion jobs.
* **Limitations:** The mock reputation service and synthetic inbox do not represent live traffic; WHOIS lookups may be incomplete in offline settings, and model false positives require human oversight.
* **Future work:** Integrate live threat intelligence feeds, incorporate analyst feedback loops for model retraining, support multi-model ensembles, and connect with orchestration tools (SOAR) for automated remediation.